# PEYNNEY'S SOLUTION IN EINSTEIN-MAXWELL-DILATON THEORIES

We consider external vacuum solutions in scalar–tensor
theories for the general class of actions

\begin{equation}
S = \int d^4x \sqrt{-g} \left[ R - 2 (\nabla \varphi)^2 - e^{-2\alpha\varphi}F^{2} \right],
\end{equation}

With the corresponding motion equations:

\begin{align*}
\nabla_\mu \left( e^{-2\alpha\phi} F^{\mu\nu} \right) = 0, \\
\nabla^2 \phi + \frac{a}{2} e^{-2\alpha\phi} F^2 = 0, \\
R_{\mu\nu} = 2 \nabla_\mu \phi \nabla_\nu \phi + 2 e^{-2\alpha\phi} F_{\mu\rho} F_\nu^{\ \rho} - \frac{1}{2} g_{\mu\nu} e^{-2\alpha\phi} F^2.
\end{align*}

Peynney (1968) provides the following static axisymmetric solution: 

\begin{align}
ds^{2} &= -\left(1-\frac{2m}{r}\right)dt^{2}
+ \exp\!\left(
-\frac{\xi^{2}m^{2}\,(r^{2}-2mr)\,\sin^{2}\theta}
{2\left(r^{2}-2mr+m^{2}\cos^{2}\theta\right)^{2}}
\right)
\left[
\left(1-\frac{2m}{r}\right)^{-1}dr^{2}
+ r^{2}d\theta^{2}
\right]
+ r^{2}\sin^{2}\theta\, d\varphi^{2}.
\tag{47}
\\[1ex]
\varphi &= \frac{m\xi}{\sqrt{r^{2}-2mr+m^{2}\cos^{2}\theta}}.
\tag{48}
\end{align}


In [1]:
%display latex

In [2]:
version()

'SageMath version 10.1, Release Date: 2023-08-20'

'SageMath version 10.1, Release Date: 2023-08-20'

In [3]:
from sage.manifolds.operators import dalembertian
from sage.manifolds.operators import laplacian
from sage.manifolds.operators import grad
from sage.symbolic.operators import FDerivativeOperator
import sage_func

In [4]:
functions_list=['U','V','X','Y','Delta','Av','Ev']
def subs_func(arg, full=True):
    subs_funcs = [(U, A), (V, B), (X, C), (Y, D),(Delta,Delta_expr),(Av,Av_expr),(Ev,Ev_expr)] \
                                        if full else [(U, A), (V, B), (X, C), (Y, D)]
    if hasattr(arg, 'expr'):
        arg = arg.expr()
    if hasattr(arg, 'apply_map')*hasattr(arg, 'display'):
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg.apply_map(lambda f: f.substitute_function(old_func, new_func).factor())
            print('{0} substituted'.format(functions_list[i])) if full else None
    elif hasattr(arg, 'apply_map'):
        return arg.apply_map(lambda f: subs_func(f).factor())
    else:
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg = arg.substitute_function(old_func, new_func).factor()
            print('{0} substituted'.format(functions_list[i])) if full else None
    print('Full Substitution : Done') if full else print('Partial Substitution : Done')
    return arg

In [5]:
M = Manifold(4, 'M', structure='Lorentzian')
print(M)

4-dimensional Lorentzian manifold M


In [6]:
XY.<t,r,th,ph> = M.chart(r"t r:(0,+oo) th:(0,pi):\theta ph:(0,2*pi):\varphi")
XY

Chart (M, (t, r, th, ph))

# I. Definition of the metric

In [7]:
U = function('U')
V = function('V')
X = function('X')
Y = function('Y')

In [8]:
g = M.metric()

g[0,0] = U(r,th)
g[1,1] = V(r,th)
g[2,2] = X(r,th)
g[3,3] = Y(r,th)

show(g.display())

g_{\mu\nu} =  [U(r, th)        0        0        0]
[       0 V(r, th)        0        0]
[       0        0 X(r, th)        0]
[       0        0        0 Y(r, th)]

In [9]:
nab = g.connection(name=r'\nabla') 

# II. The vector potential

In [10]:
m, xi, a, B = var('m xi alpha B')
assume(m >= 0, 'real')
assume(xi > 0, 'real')

Delta = function('Delta')

pot_vec = M.tensor_field(0,1,name='A')
pot_vec[0]=0
pot_vec[1]=0
pot_vec[2]=0
pot_vec[3]= - 2 / (1+alpha**2) / B / Delta(r,th)

show(pot_vec.display())

A = -2/((alpha^2 + 1)*B*Delta(r, th)) dph

In [11]:
DF = nab(pot_vec) ; DF

Tensor field \nabla(A) of type (0,2) on the 4-dimensional Lorentzian manifold M

# III. Definition of the EM tensor

In [12]:
F = diff(pot_vec)
F.set_name('F')
Fuu = F.up(g)
F.display()

F = 2*d(Delta)/dr/((B*alpha^2 + B)*Delta(r, th)^2) dr∧dph + 2*d(Delta)/dth/((B*alpha^2 + B)*Delta(r, th)^2) dth∧dph

In [13]:
epsilon_3D = M.tensor_field(0,3, antisymmetric=True, name=r'\epsilon_{c}') 
epsilon_3D[1,2,3] = 1
epsilon_3D[2,3,1] = 1
epsilon_3D[3,1,2] = 1
epsilon_3D[3,2,1] = -1
epsilon_3D[2,1,3] = -1
epsilon_3D[1,3,2] = -1
epsilon_3D.display()

\epsilon_{c} = dr⊗dth⊗dph - dr⊗dph⊗dth - dth⊗dr⊗dph + dth⊗dph⊗dr + dph⊗dr⊗dth - dph⊗dth⊗dr

In [14]:
B_mag = M.tensor_field(1,0)  
for i in [1, 2, 3]: 
    expr = 1/2 * sum(epsilon_3D[i,j,k] * F[j,k] for j in [1,2,3] for k in [1,2,3])
    B_mag[i] = expr.expr().factor() 
show(LatexExpr(r'B^{\mu} = '),B_mag[:])

B^{\mu} =  [0,
 2*d(Delta)/dth/((alpha^2 + 1)*B*Delta(r, th)^2),
 -2*d(Delta)/dr/((alpha^2 + 1)*B*Delta(r, th)^2),
 0]

# IV. Variables and functions

In [15]:
Av = function('A_v')
Ev = function('E_v')

In [16]:
Av_expr(r) = (1- 2*m /r)
Ev_expr(r,th) = exp(- (xi**2 * m**2 * r**2 * Av_expr(r) * sin(th)**2) / (2 * (r**2*Av_expr(r) + m**2 * cos(th)**2)**2))
old_Phi = M.scalar_field({XY:m*xi / sqrt(2*(r**2*Av_expr(r) + m**2 * cos(th)**2))}, name=r'\varphi')
old_Psi = exp(2*alpha*old_Phi)
Delta_expr(r, th) = 1 + (1+ alpha**2) / 4 * B**2 * (r*sin(th))**2 * old_Psi.expr()

In [17]:
show(LatexExpr(r'\Delta = '), Delta_expr(r, th))
show(LatexExpr(r'A_v(r) = '), Av_expr(r), LatexExpr(r', \quad\quad E_v(r,\theta) = '), Ev_expr(r,th))
show(LatexExpr(r"\phi_{0} = "), old_Phi.expr())
show(LatexExpr(r"\Psi_{0} = e^{2\alpha\phi_{0}} = "), old_Psi.expr())

\Delta =  1/4*(alpha^2 + 1)*B^2*r^2*e^(sqrt(2)*alpha*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))*sin(th)^2 + 1

A_v(r) =  -2*m/r + 1 , \quad\quad    E_v(r,\theta) =  e^(1/2*m^2*r^2*xi^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2)

\phi_{0} =  m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1))

\Psi_{0} = e^{2\alpha\phi_{0}} =  e^(sqrt(2)*alpha*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

# V. Metric and potential vector expressions

In [18]:
A(r,th) = - Av(r) * Delta(r,th)**(2/(1+alpha**2))
B(r,th) = Ev(r,th) / Av(r) * Delta(r,th)**(2/(1+alpha**2))
C(r,th) = r**2 * Ev(r,th) * Delta(r,th)**(2/(1+alpha**2))
D(r,th) =  (r*sin(th))**2 / Delta(r,th)**(2/(1+alpha**2))
g_t = g.copy(name ='g')
g_t.apply_map(lambda f:f.substitute_function(U, A).substitute_function(V, B).substitute_function(X, C).substitute_function(Y, D))

show(LatexExpr(r"A_{\mu\nu} = 0 \rightarrow A'_{\mu\nu} ="), pot_vec.display())
show(LatexExpr(r'g_{\mu\nu} = '), g_t.display())
show(LatexExpr(r'\sqrt{-g} = '),subs_func((-g.det().expr())^(1/2), full=False).canonicalize_radical())

A_{\mu\nu} = 0 \rightarrow A'_{\mu\nu} = A = -2/((alpha^2 + 1)*B*Delta(r, th)) dph

g_{\mu\nu} =  g = -Delta(r, th)^(2/(alpha^2 + 1))*A_v(r) dt⊗dt + Delta(r, th)^(2/(alpha^2 + 1))*E_v(r, th)/A_v(r) dr⊗dr + r^2*Delta(r, th)^(2/(alpha^2 + 1))*E_v(r, th) dth⊗dth + r^2*sin(th)^2/Delta(r, th)^(2/(alpha^2 + 1)) dph⊗dph

Partial Substitution : Done


\sqrt{-g} =  r^2*Delta(r, th)^(2/(alpha^2 + 1))*E_v(r, th)*abs(sin(th))

In [19]:
g_nod = g.copy()
g_nod = subs_func(g_nod)
nod_ratio = (g_nod[3,3]/(sin(th)**2*g_nod[2,2])).factor()
show(LatexExpr(r"\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = " + latex(limit(nod_ratio.expr(), th=0).canonicalize_radical())))

U substituted
V substituted
X substituted
Y substituted
Delta substituted
Av substituted
Ev substituted
Full Substitution : Done


\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = 1

# VI. The field equations

## VI.1. Verification of the Maxwell equations $\bigtriangledown_{\mu}(e^{-2\alpha\Phi}F^{\mu\nu})=0$

In [20]:
Phi = M.scalar_field({XY:-alpha/ (1+alpha**2) * log(Delta(r,th))+old_Phi.expr()}, name=r'\phi') 
Psi = M.scalar_field({XY:old_Psi.expr() / (Delta(r,th))**(2*alpha**2/(1+alpha**2))}, name=r'\Psi')

show(LatexExpr(r'\varphi = '), old_Phi.expr(), LatexExpr(r"\rightarrow \varphi' = "), Phi.expr())
show(LatexExpr(r'\Psi = '), old_Psi.expr(), LatexExpr(r"\rightarrow \Psi' = "), Psi.expr())

\varphi =  m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) \rightarrow \varphi' =  m*xi/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) - alpha*log(Delta(r, th))/(alpha^2 + 1)

\Psi =  e^(sqrt(2)*alpha*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)) \rightarrow \Psi' =  e^(sqrt(2)*alpha*m*xi/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/Delta(r, th)^(2*alpha^2/(alpha^2 + 1))

In [21]:
eq1 = nab(Fuu / Psi)
eq = eq1['^a._a']

In [22]:
eq.apply_map(lambda f:f.factor())
eq = subs_func(eq)
show('Substitution: done')

U substituted
V substituted
X substituted
Y substituted
Delta substituted
Av substituted
Ev substituted
Full Substitution : Done


'Substitution: done'

In [23]:
show('Verification: ', LatexExpr(r'\bigtriangledown_{\mu}(e^{-2\alpha\Phi}F^{\mu\nu})=0: '), eq == 0)

'Verification: ' \bigtriangledown_{\mu}(e^{-2\alpha\Phi}F^{\mu\nu})=0:  True

## VI. 2. Verification of the equation $\bigtriangledown^{2}\Phi+\frac{\alpha}{2}e^{-2\alpha\Phi}F^{2}=0$ 

In [24]:
g = subs_func(g,full=False)

Partial Substitution : Done


In [25]:
F2 = F['_ab']*Fuu['^ab']
eq2_1 = Phi.dalembertian()
eq2_2 = (a/2)*F2 / Psi 
eq2 = eq2_1 + eq2_2

In [26]:
eq_num = numerator(eq2.expr().factor())
eq_num = subs_func(eq_num)

U substituted
V substituted
X substituted
Y substituted
Delta substituted
Av substituted
Ev substituted
Full Substitution : Done


In [27]:
LateX = LatexExpr(r'\bigtriangledown^{2}\Phi+\frac{\alpha}{2}e^{-2\alpha\Phi}F^{2}=0 ')
show('Verification: ', LateX, LatexExpr(r'\quad'), eq_num.is_zero())

'Verification: ' \bigtriangledown^{2}\Phi+\frac{\alpha}{2}e^{-2\alpha\Phi}F^{2}=0  \quad True

## VI.3 Verification of the main motion equation $R_{\mu\nu}=2\triangledown_{\mu}\Phi\triangledown_{\nu}\Phi+\frac{T_{\mu\nu}}{\Psi}$

In [28]:
Fud = F.up(g,0)
T = 2*(F['_k.']*Fud['^k_.'] - 1/4*F2*g)
UU=T / Psi

In [29]:
nab_phi = nab(Phi)
S = 2*nab_phi*nab_phi
RHS = UU + S

In [30]:
ER_ricc = g.ricci()

In [31]:
RHS = subs_func(RHS, full=False)

Partial Substitution : Done


In [32]:
eq3 = ER_ricc - RHS

In [33]:
eq3.apply_map(lambda f :f.substitute_function(Delta,Delta_expr).substitute_function(Av,Av_expr)\
                  .substitute_function(Ev,Ev_expr).factor())

In [34]:
LateX = LatexExpr(r'R_{\mu\nu}=2\triangledown_{\mu}\Phi\triangledown_{\nu}\Phi+\frac{T_{\mu\nu}}{\Psi} ')
show('Verification: ',LateX, LatexExpr(r'\quad'), eq3.is_zero())

'Verification: ' R_{\mu\nu}=2\triangledown_{\mu}\Phi\triangledown_{\nu}\Phi+\frac{T_{\mu\nu}}{\Psi}  \quad True

# WE CONCLUDE THAT THE PENNEY SOLUTION EMBEDDED INTO A MAGNETIC FIELD SATISFIES THE EINSTEIN-MAXWELL-DILATON EQUATIONS FOR ARBITRARY COUPLING